In [1]:
import math 
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
class Value:
    index = 0
    def __init__(self, data, _children=(), _op='', label=None):
        self.data = data
        if label:
            self.label = label
        else :
            self.label =f"v_{Value.index}"
            Value.index += 1
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set[Value](_children)
        self._op = _op

    def __repr__(self):
        #return f"Value({self.label}: data={self.data}), grad={round(self.grad,10)}, op={self._op}"
        return f"Value({self.label}: data={self.data}), grad={self.grad}, op={self._op}"

    def __add__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out =  Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + other*(-1)

    def __rsub__(self, other):
        return self*(-1) + other

    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out        

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float are supported"
        out =  Value(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += out.grad* other*self.data**(other-1)
        out._backward = _backward
        return out   

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def tanh(self):
        out =  Value(math.tanh(self.data), (self,), 'tanh')
        def _backward():
            self.grad += out.grad/(math.cosh(self.data)**2)
        out._backward = _backward
        return out   

    def exp(self):
        out =  Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.grad*math.exp(self.data)
        out._backward = _backward
        return out   

    def print_chain(self):
        if self._prev:
            for p in self._prev:
                p.print_chain()
        print(self)

    
    def backward(self) -> None:
        self.grad = 1.0

        # topological sort the nodes
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # backward pass
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
                    



In [35]:
"""
# input layer
a = Value(3.0, label='a')
b = a.tanh()
b.label = "b"
b.backward()
print("--------------------------------")
b.print_chain()


x = Value(3.0, label='x')
tmp = (-1)*x
# y = x.exp() - tmp.exp()  OK
# y = x.exp() + tmp.exp()  OK
# y =(tmp**2)*(tmp**(-1)) OK
y = (x.exp() - tmp.exp()) / (x.exp() + tmp.exp())  # OK
y.label = "y"
y.backward()
print("--------------------------------")
y.print_chain()



a = Value(3.0, label='a')
b = 2*a
b.label = "b"
c = (3*b) - (2*b)
c.label = "c"
c.backward()
print("--------------------------------")
c.print_chain()


a = Value(3.0, label='a')
b = a+a; b.label = "b"
b.backward()
print("--------------------------------")
b.print_chain()

a = Value(-2.0, label='a')
b = Value(3.0, label='b')
d = a*b; d.label = "d"
e = a + b; e.label = "e"
f = d*e; f.label = "f"
f.backward()
print("--------------------------------")
f.print_chain()
"""

x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

b = Value(6.8813735870195432, label='b')

n = x1*w1 + x2*w2 + b; n.label = "n"
o = n.tanh()
o.backward()
print("--------------------------------")
o.print_chain()













--------------------------------
Value(b: data=6.881373587019543), grad=0.4999999999999999, op=
Value(x2: data=0.0), grad=0.4999999999999999, op=
Value(w2: data=1.0), grad=0.0, op=
Value(v_119: data=0.0), grad=0.4999999999999999, op=*
Value(x1: data=2.0), grad=-1.4999999999999996, op=
Value(w1: data=-3.0), grad=0.9999999999999998, op=
Value(v_118: data=-6.0), grad=0.4999999999999999, op=*
Value(v_120: data=-6.0), grad=0.4999999999999999, op=+
Value(n: data=0.8813735870195432), grad=0.4999999999999999, op=+
Value(v_122: data=0.7071067811865476), grad=1.0, op=tanh


In [ ]:
import torch
x1 = torch.Tensor([2]).double()                         ; x1.requires_grad = True
x2 = torch.Tensor([0]).double()                         ; x2.requires_grad = True   

w1 = torch.Tensor([-3]).double()                        ; w1.requires_grad = True
w2 = torch.Tensor([1]).double()                         ; w2.requires_grad = True

b = torch.Tensor([6.8813735870195432]).double()         ; b.requires_grad = True

n = x1*w1 + x2*w2 + b; n.label = "n"
o = torch.tanh(n)
print(o.data.item())

o.backward()
print("--------------------------------")
print(x1.grad.item())
print(x2.grad.item())
print(w1.grad.item())
print(w2.grad.item())
print(b.grad.item())




0.7071066904050358
--------------------------------
-1.5000003851533106
0.5000001283844369
1.0000002567688737
0.0
0.5000001283844369


In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((xi*wi for xi,wi in zip(x, self.w)), self.b)
        out = act.tanh()
        return out

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x



mpl = MLP(3, [4,4,1])
x = [2,3,-1]
print(mpl(x))



[Value(v_588: data=0.9524890388911927), grad=0.0, op=tanh]
